# Task 2 — DeiT: does seeing more pixels help? (Colab runner)

Sets up the repo on Colab and runs every experiment. Run the cells top to bottom.

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the setup cells. They clone the repo, install `einops`, and download Imagewoof (~1.3 GB, under a minute).
3. Run whichever experiments you want. Each one trains its own copy of the pretrained model and saves its accuracy to `results/`. The last cell prints them all.

Each @384 run takes roughly 25–35 min on a T4 (11 passes at 2–3 min each). The first experiment that trains asks to **mount Google Drive**: checkpoints for C and E go to `MyDrive/deit_seminar_task2`. The trained C and E checkpoints committed in `labs2/checkpoints/` are copied there first, so C/E load them instead of retraining (delete them from Drive to retrain).

## Setup

In [ ]:
# The repo and branch to run. If the repo is private, use https://<token>@github.com/... instead.
REPO = "https://github.com/typsoon/pattern-recognition.git"
BRANCH = "main"
REPO_DIR = "/content/pattern-recognition"

In [ ]:
import os, sys

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {BRANCH} && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull --ff-only

os.chdir(f"{REPO_DIR}/labs2")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
!git log --oneline -1

In [ ]:
# torch, torchvision and timm are preinstalled on Colab; einops is the only extra dependency
!pip install -q einops
!nvidia-smi -L

In [ ]:
# Downloads Imagewoof if missing, builds cfg / to_imagenet, defines the fixed evaluate()
from common import evaluate, load_pretrained, save_result

model = load_pretrained()   # the given DeiT-S distilled @224; every experiment copies it

## Baseline @224

Should print **97.20%**. If not, something in the setup went wrong.

In [ ]:
baseline_acc = evaluate(model, 224)
print(f"baseline DeiT-S distilled @224: top-1 {baseline_acc:.2%}")
save_result("baseline_224", baseline_acc)

## 0-shot @384 per interpolation mode

**Author: Piotr** (`exp_a_zero_shot.py`). No training, about a minute per mode.

In [ ]:
from exp_a_zero_shot import MODES, run_zero_shot

zero_shot = {mode: run_zero_shot(model, 384, mode)[1] for mode in MODES}

## Fine-tuning control @224

**Author: Piotr** (`finetune_pre_resizing.py`). The same budget at the original resolution, to compare the @384 runs against.

In [ ]:
from finetune_pre_resizing import run_control

model_control, acc_control = run_control(model, 224)

## Linear probe (B1, B2) and layer freezing (D) @384

**Author: Gabriela** (`exp_b_linear_probe.py`, `exp_d_layer_freezing.py`)

In [ ]:
from exp_b_linear_probe import run_b1, run_b2
from exp_d_layer_freezing import run_d

In [ ]:
model_b1, acc_b1 = run_b1(model)

In [ ]:
model_b2, acc_b2 = run_b2(model)

In [ ]:
model_d, acc_d = run_d(model)

## Full fine-tuning (C) and progressive resolution (E) @384

**Author: Vitaliy** (`exp_c_full_finetune.py`, `exp_e_progressive.py`). Both load a saved checkpoint from Drive instead of retraining if one exists. The cell below copies the committed checkpoints from `checkpoints/` to Drive (files already on Drive are kept).

In [ ]:
import shutil

from training import SAVE_DIR
from exp_c_full_finetune import run_c
from exp_e_progressive import run_e

# run_c / run_e look for checkpoints in SAVE_DIR (Drive on Colab), not in the repo's checkpoints/
for name in os.listdir("checkpoints"):
    dst = os.path.join(SAVE_DIR, name)
    if not os.path.exists(dst):
        shutil.copy(os.path.join("checkpoints", name), dst)
        print(f"copied checkpoints/{name} -> {dst}")

In [ ]:
model_c, acc_c = run_c(model)

In [ ]:
model_e, acc_e = run_e(model)

## Summary

Every saved result against the @224 baseline. `results/` lives in the Colab VM and is lost when the runtime resets, so copy the numbers out (or run the cell below the summary).

In [ ]:
!python summary.py

In [ ]:
# Optional: keep the results JSONs on Drive next to the checkpoints
!mkdir -p /content/drive/MyDrive/deit_seminar_task2/results && cp results/*.json /content/drive/MyDrive/deit_seminar_task2/results/